<a href="https://colab.research.google.com/github/RobJavVar/DataSciencePsychNeuro/blob/master/ExerciseSubmissions/17_regularized-regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 17: Regularized regression

This homework assignment is designed to give you an intuition as an interesting property of regularization in the context of ultra-high dimensional statistical problems.

You won't need to load in any data for this homework.

---
## 1. Simulating & visualizing data (2 points)

We are going to be looking at what happens in the context where $p>n$. In order to have total control over our data, we will use simulations for this homework. First, we will need to load the `glmnet`, `tidyverse`, and `ggplot2` libraries for this assignment.


In [ ]:
library(glmnet)
library(tidyverse)
library(ggplot2)


We are going to generate a data set with complex structure and try to recover it using polynomial models. For simplicity sake, use the following code to produce a response variable, $y$ that has complex structure.

*Hint: Look up what a cosine function looks like if you need a reminder.*

In [ ]:
# Generate data
set.seed(121)
sigma_noise = .5
x=seq(-9,9,by=.18)
n=length(x)
y = 0.1*x + cos(x) + cos(x/20)+rnorm(n,sd=sigma_noise)

(a) Break the data into a training set (n=50) and test set (n=51) using the `sample` function to randomly select subsets of x and y.  Make a separate data frame for the training and test data.

(**Note**: *Do not* just take the first 50 observations to be the training set and last 51 observations to be the test set.)

In [ ]:
train_idx <- sample(1:101, 50, replace = F)
y_train <- y[train_idx]
x_train <- x[train_idx]

y_test <- y[-train_idx]
x_test <- x[-train_idx]

df_train <- data.frame(
    x = x_train,
    y = y_train
)

df_test <- data.frame(
    x = x_test,
    y = y_test
)


(b) Plot the training data ($x$ \& $y$). Describe the relationship that you see in the training data.

In [ ]:
ggplot(data = df_train, aes(x = x, y = y)) + geom_point() + geom_smooth(method = "lm")
ggplot(data = df_test, aes(x = x, y = y)) + geom_point() + geom_smooth(method = "lm")


How would you describe the relationship between $x$ and $y$ based on this plot?

> There appears to be a positive relationship between x and y. The confidence interval is wide but it may still be significant.
>

---
## 2. Bias-variance tradeoff: polynomial regression (4 points)

Recall that in polynomial regression we increase model complexity by expanding $x$ out to the power $k$ (which we call degree).

$$
Y = \hat{\beta}_0 + \sum_{j=1}^K \hat{\beta}_j X^j
\;=\; \operatorname{poly}(X, K)
$$

(a) Fit a 2nd degree polynomial regression model to the training data. Plot the results.


*Hint: Use the* `help` *function to see how to use the* `stat_smooth()` *and* `poly()` *functions.*

In [ ]:
ggplot(data = df_train, aes(x = x, y = y)) + geom_point() + stat_smooth(method = "lm", formula = y ~ poly(x, 2), color = "red")


How well does this 2nd degree polynomial model qualitatively fit the data? Could it do better?

> It doesn't look that different from the linear model, it doesn't seem worth it to do second degree. The error bars are still pretty big. 
>

(b) Fit a 12th degree polynomial to the data. Does this do qualitatively better or worse than the 2nd degree model?

In [ ]:
ggplot(data = df_train, aes(x = x, y = y)) + geom_point() + stat_smooth(method = "lm", formula = y ~ poly(x, 12), color = "red")


> The error bars are smaller, but there is no longer a clear relationship between the variables. The line is super squiggly, and it looks way overfitted.
>

(c) Modify the loop below to estimate the bias-variance tradeoff as model complexity (i.e., degree of the polynomial model, $k$) increases from 2 to 50. Use the training data to fit the model and test data to evaluate its predictive accuracy.

Visualize your results by plotting the *median* squared error for the training data and test data as a function of polynomial degree.


(**Note**: We are using median accuracies here because there are often 1 or 2 outlier values in the higher degree polynomial models that can throw off the accuracy estimates).

In [ ]:
degree <- seq(2, 50)

train_rss <- matrix(data = NA, nrow = length(degree), ncol = 1)
test_rss  <- matrix(data = NA, nrow = length(degree), ncol = 1)

for (i in 1:length(degree)) {
    k <- degree[i]
    
    formula_str <- paste0("y ~ poly(x, ", k, ", raw = TRUE)")
    train_model <- glm(data = df_train, formula = as.formula(formula_str))
    
    pred_train <- predict(train_model, newdata = df_train)
    pred_test  <- predict(train_model, newdata = df_test)
  
    train_rss[i] <- median((df_train$y - pred_train)^2)
    test_rss[i]  <- median((df_test$y  - pred_test)^2)
}

results_df <- data.frame(
  Degree = rep(degree, 2),
  Error = c(train_rss, test_rss),
  Dataset = rep(c("Train", "Test"), each = length(degree))
)

ggplot(results_df, aes(x = Degree, y = Error, color = Dataset)) +
  geom_line(linewidth = 1) +
  geom_point() +
  scale_color_manual(values = c("Train" = "blue", "Test" = "red"))


What do you see as $k$ increase?

> (From zooming in) at first, both the train and test accuracy improves. After about the 15gh degree, the test accuracy begins to get worse, though the training accuracy continues to improve.
> Then, after k = 35 the error begins to really diverge the test set has huge error levels. It also appears at high k the training error flattens out, hitting a sort of minimum.
> Note: I could not get it to work for some reason with k = 50, i asked gemini for help and tried a bunch of stuff and finally got it. I had to change the loop - I'm not entirely sure why it didn't work with k in degree but alas...

(d) Now copy the code above and let's see what happens when we go beyond $p=n$ (remember, in this case $k=p$). Test polynomial models up to $k=150$. Visualize your results by plotting the *median* squared error for the training data and test data as a function of polynomial degree.

Use the `geom_vline()` function in `ggplot` to draw a vertical line where $k=n$ (here $n$ is the number of observations in the training set). This will make it clear where we cross the threshold for finding *unique* solutions in our data.



In [ ]:
degree1 <- seq(2, 150)

train_rss1 <- matrix(data = NA, nrow = length(degree), ncol = 1)
test_rss1  <- matrix(data = NA, nrow = length(degree), ncol = 1)

for (i in 1:length(degree1)) {
    k <- degree1[i]
    
    formula_str <- paste0("y ~ poly(x, ", k, ", raw = TRUE)")
    train_model <- glm(data = df_train, formula = as.formula(formula_str))
    
    pred_train <- predict(train_model, newdata = df_train)
    pred_test  <- predict(train_model, newdata = df_test)
  
    train_rss1[i] <- median((df_train$y - pred_train)^2)
    test_rss1[i]  <- median((df_test$y  - pred_test)^2)
}

results_df1 <- data.frame(
  Degree = rep(degree1, 2),
  Error = c(train_rss1, test_rss1),
  Dataset = rep(c("Train", "Test"), each = length(degree1))
)

ggplot(results_df1, aes(x = Degree, y = Error, color = Dataset)) +
  geom_line(linewidth = 1) +
  geom_point() +
  scale_color_manual(values = c("Train" = "blue", "Test" = "red")) +
  geom_vline(xintercept = 50)



What do you see as $k$ gets larger than $n$?

> The model still runs if you don't do orthagonal polynomials. However, shortly after k = 50 the errors diverge extremely, and they both seem to hit some sort of maximum bound where they remain until k = 150.
>

---
## 3. Applying regularization to the model fits (2 points)

Repeat the previous bias-variance tradeoff test, going up to $k=150$, but now use ridge regression with a sparsity parameter of $\lambda=0.00005$. Plot your results the same way as last time.

In [ ]:
# Now do the variance-bias trade off analysis using ridge regression
lambda=0.00005
degree = seq(2,150)

rm(train_rss, test_rss)
train_rss = matrix(data=NA,nrow=length(degree),ncol=1)
test_rss = matrix(data=NA,nrow=length(degree),ncol=1)

for (i in 1:length(degree)) {
    k <- degree[i]

    x_train_mat <- poly(df_train$x, k, raw = TRUE)
    x_test_mat  <- poly(df_test$x, k, raw = TRUE)
    
    train_model <- glmnet(x = x_train_mat, y = df_train$y, alpha = 0, lambda = lambda)    
    
   pred_train <- as.numeric(predict(train_model, newx = x_train_mat))
    pred_test  <- as.numeric(predict(train_model, newx = x_test_mat))
  
    train_rss[i] <- median((df_train$y - pred_train)^2)
    test_rss[i]  <- median((df_test$y  - pred_test)^2)
}

results_df <- data.frame(
  Degree = rep(degree, 2),
  Error = c(train_rss, test_rss),
  Dataset = rep(c("Train", "Test"), each = length(degree))
)

ggplot(results_df, aes(x = Degree, y = Error, color = Dataset)) +
  geom_line(linewidth = 1) +
  geom_point() +
  scale_color_manual(values = c("Train" = "blue", "Test" = "red"))+
  geom_vline(xintercept = 50)


What happens now when $k$ gets larger than $n$?

> The errors for the train and test remain constant, they don't increase as before.
>

---
## 4. Reflection (2 points)

The simulations above should have shown that, when applying a regularization (i.e., a sparsity constraint), the behavior of the bias-variance tradeoff changes. Explain why this happens.

> In the standard regularization the model is just trying to fit the test data as well as possible, so it might have crazy large coefficients for the input variables. But this will inevitably cause very high bias, so the test error keeps increasing and increasing. This is why the bias keeps increasing after a certain point.
> In the ridge regression, the sparsity constraint tells the model to keep the coefficient beta values close to zero, unlike the standard model. So there won't be any values that are too crazy and increasing the bias as much when k increases. Therefore the fit for test data doesn't increase as much.

---
## Bonus (1 extra credit point)
Recall that the $p=n$ threshold defines the limit for finding a *unique* solution to $Y=F(X)$ (i.e., there is only one combination of regression coefficients that is *best* at explaining variance in $Y$). With this in mind, what is regularization doing that works around this upper limit?

> It is using a lambda value as another decision factor, so we can still find a unique solution. Where there may be a bunch of solutions that minimize the error, the lambda gives us a new way to pick the best model - the one with betas closest to zero.
> 

**DUE:** 5pm EST, April 7, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> I had to ask gemini how to do the plotting and for loop because i was getting weird errors